In [1]:
import gc
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

RANDOM_STATE = 10
TEST_SIZE = 0.20
OUTPUT_DIR = Path("autoencoder_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
APO_FILE = "all-apo-dist.csv"

HOLO_FILES = {
    "PAKPV": "4zlo-holo-merged_dist.csv",
    "PAKOQ": "4zji-holo-merged_dist.csv",
    "PAKOR": "4zjj-holo-merged_dist.csv",
}

print("Apo file:", APO_FILE)
print("Holo files:")
for system, filename in HOLO_FILES.items():
    print(f"  {system}: {filename}")


Apo file: all-apo-dist.csv
Holo files:
  PAKPV: 4zlo-holo-merged_dist.csv
  PAKOQ: 4zji-holo-merged_dist.csv
  PAKOR: 4zjj-holo-merged_dist.csv


In [4]:
def read_distance_csv(filename):
    """Read a distance-feature CSV and ensure that all columns are numeric."""
    data = pd.read_csv(filename)

    if data.empty:
        raise ValueError(f"{filename} contains no rows.")

    non_numeric = data.columns[~data.dtypes.apply(lambda dtype: np.issubdtype(dtype, np.number))]
    if len(non_numeric) > 0:
        raise TypeError(
            f"{filename} contains non-numeric columns: {list(non_numeric)}. "
            "Remove labels or metadata before reconstruction."
        )

    if data.isna().any().any():
        raise ValueError(f"{filename} contains NaN values.")

    return data


def check_feature_compatibility(reference, query, query_name):
    """Verify that the holo data have exactly the same features and order as apo."""
    if query.shape[1] != reference.shape[1]:
        raise ValueError(
            f"{query_name}: expected {reference.shape[1]} features, "
            f"but found {query.shape[1]}."
        )

    if not query.columns.equals(reference.columns):
        missing = reference.columns.difference(query.columns).tolist()
        extra = query.columns.difference(reference.columns).tolist()
        raise ValueError(
            f"{query_name}: column names/order do not match the apo dataset. "
            f"Missing examples: {missing[:5]}; extra examples: {extra[:5]}"
        )


def save_dio_histogram(dio_df, output_file, title):
    """Save the normalized DIO histogram without holding an unnecessary copy."""
    values = dio_df.to_numpy().ravel()

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(
        values,
        weights=np.ones(values.size, dtype=float) / values.size,
        bins=np.arange(-10.5, 10.5, 1.0),
        alpha=0.9,
        rwidth=0.5,
    )
    ax.set_xlabel("DIO = input distance - reconstructed distance")
    ax.set_ylabel("Fraction")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_file, dpi=300, bbox_inches="tight")
    plt.close(fig)


def reconstruct_and_write(model, data, system_name, subset_name):
    """Reconstruct a dataset, calculate DIO, and write CSV and PNG outputs."""
    reconstruction = model.predict(data)
    dio = pd.DataFrame(
        data.to_numpy() - reconstruction,
        index=data.index,
        columns=data.columns,
    )

    csv_file = OUTPUT_DIR / f"dio_{system_name}_{subset_name}.csv"
    png_file = OUTPUT_DIR / f"{system_name}_{subset_name}_dio.png"

    dio.to_csv(csv_file, index=False)
    save_dio_histogram(
        dio,
        png_file,
        title=f"{system_name}: {subset_name} DIO distribution",
    )

    summary = {
        "system": system_name,
        "subset": subset_name,
        "n_frames": len(data),
        "n_features": data.shape[1],
        "mean_DIO": float(dio.to_numpy().mean()),
        "mean_abs_DIO": float(np.abs(dio.to_numpy()).mean()),
        "rmse": float(np.sqrt(np.mean(np.square(dio.to_numpy())))),
        "csv_file": str(csv_file),
        "histogram_file": str(png_file),
    }

    del reconstruction, dio
    gc.collect()
    return summary


In [5]:
apo_df = read_distance_csv(APO_FILE)

print("Reference apo shape:", apo_df.shape)
print("Number of reference features:", apo_df.shape[1])


Reference apo shape: (5000, 41616)
Number of reference features: 41616


In [6]:
MODEL_FILE = "allapo_trained_autoencoder.joblib"

reg = joblib.load(MODEL_FILE)

print("Loaded model:", MODEL_FILE)
print("Model expects features:", reg.n_features_in_)
print("Reference apo features:", apo_df.shape[1])

if reg.n_features_in_ != apo_df.shape[1]:
    raise ValueError(
        f"Model expects {reg.n_features_in_} features, "
        f"but the reference apo dataset contains {apo_df.shape[1]}."
    )

print("The saved model and reference apo dataset are compatible.")

Loaded model: allapo_trained_autoencoder.joblib
Model expects features: 41616
Reference apo features: 41616
The saved model and reference apo dataset are compatible.


In [7]:
all_summaries = []

for system_name, holo_file in HOLO_FILES.items():
    print("\n" + "=" * 80)
    print(f"Processing {system_name}: {holo_file}")

    holo_df = read_distance_csv(holo_file)
    check_feature_compatibility(apo_df, holo_df, system_name)
    print("Holo shape:", holo_df.shape)

    holo_remaining, holo_test = train_test_split(
        holo_df,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        shuffle=True,
    )


    all_summaries.append(
        reconstruct_and_write(
            model=reg,
            data=holo_test,
            system_name=system_name,
            subset_name="heldout_test",
        )
    )

    all_summaries.append(
        reconstruct_and_write(
            model=reg,
            data=holo_df,
            system_name=system_name,
            subset_name="whole_dataset",
        )
    )


    all_summaries.append(
        reconstruct_and_write(
            model=reg,
            data=holo_remaining,
            system_name=system_name,
            subset_name="remaining_subset",
        )
    )

    del holo_df, holo_remaining, holo_test
    gc.collect()

print("\nAll holo systems have been processed with the same apo-trained model.")



Processing PAKPV: 4zlo-holo-merged_dist.csv
Holo shape: (15000, 41616)

Processing PAKOQ: 4zji-holo-merged_dist.csv
Holo shape: (15000, 41616)

Processing PAKOR: 4zjj-holo-merged_dist.csv
Holo shape: (15000, 41616)

All holo systems have been processed with the same apo-trained model.


In [8]:
summary_df = pd.DataFrame(all_summaries)
summary_file = OUTPUT_DIR / "reconstruction_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Summary written to:", summary_file)
summary_df


Summary written to: autoencoder_outputs/reconstruction_summary.csv


,system,subset,n_frames,n_features,mean_DIO,mean_abs_DIO,rmse,csv_file,histogram_file
0,PAKPV,heldout_test,3000,41616,-0.466658,1.402299,2.043901,autoencoder_outputs/dio_PAKPV_heldout_test.csv,autoencoder_outputs/PAKPV_heldout_test_dio.png
1,PAKPV,whole_dataset,15000,41616,-0.466569,1.401773,2.043119,autoencoder_outputs/dio_PAKPV_whole_dataset.csv,autoencoder_outputs/PAKPV_whole_dataset_dio.png
2,PAKPV,remaining_subset,12000,41616,-0.466547,1.401641,2.042924,autoencoder_outputs/dio_PAKPV_remaining_subset...,autoencoder_outputs/PAKPV_remaining_subset_dio...
3,PAKOQ,heldout_test,3000,41616,-0.461373,1.563026,2.478653,autoencoder_outputs/dio_PAKOQ_heldout_test.csv,autoencoder_outputs/PAKOQ_heldout_test_dio.png
4,PAKOQ,whole_dataset,15000,41616,-0.461560,1.563383,2.478371,autoencoder_outputs/dio_PAKOQ_whole_dataset.csv,autoencoder_outputs/PAKOQ_whole_dataset_dio.png
5,PAKOQ,remaining_subset,12000,41616,-0.461606,1.563472,2.478301,autoencoder_outputs/dio_PAKOQ_remaining_subset...,autoencoder_outputs/PAKOQ_remaining_subset_dio...
6,PAKOR,heldout_test,3000,41616,-0.460311,1.607105,2.519000,autoencoder_outputs/dio_PAKOR_heldout_test.csv,autoencoder_outputs/PAKOR_heldout_test_dio.png
7,PAKOR,whole_dataset,15000,41616,-0.460272,1.608168,2.521254,autoencoder_outputs/dio_PAKOR_whole_dataset.csv,autoencoder_outputs/PAKOR_whole_dataset_dio.png
8,PAKOR,remaining_subset,12000,41616,-0.460262,1.608433,2.521817,autoencoder_outputs/dio_PAKOR_remaining_subset...,autoencoder_outputs/PAKOR_remaining_subset_dio...
